In [2]:

import tkinter as tk
from tkinter import ttk, messagebox
import os #sirve para que Python pueda trabajar con archivos y carpetas del sistema operativo

ARCHIVO = "estudios.txt"# definir donde guardar la info
materias = []

def cargar():
    if not os.path.exists(ARCHIVO):
        return []
    datos = []
    try:
        with open(ARCHIVO, "r", encoding="utf-8") as f:
            for linea in f:
                partes = linea.strip().split("|")

                if len(partes) == 5:
                    datos.append({  "materia": partes[0], "profesor": partes[1],
                    "nota": float(partes[2]), "fecha": partes[3], "estado": partes[4]
                    })

    except (ValueError, OSError):
        messagebox.showerror(
            "Error",
            "No se pudieron cargar los datos."
        )

    return datos


def guardar():
    try:
        with open(ARCHIVO, "w", encoding="utf-8") as f:
            for m in materias:
                f.write(   f"{m['materia']}|{m['profesor']}|"   f"{m['nota']}|{m['fecha']}|{m['estado']}\n"
                )

    except OSError:
        messagebox.showerror(
            "Error",
            "No se pudieron guardar los datos."
        )


def limpiar():
    for entrada in (
        entrada_materia,
        entrada_profesor,
        entrada_nota,
        entrada_fecha
    ):
        entrada.delete(0, tk.END)

    estado.set("Pendiente")


def actualizar(lista=None):
    tabla.delete(*tabla.get_children())

    for m in lista if lista is not None else materias:
        tabla.insert(
            "",
            tk.END,
            values=(  m["materia"],  m["profesor"],  m["nota"],  m["fecha"],  m["estado"]
            )
        )

    promedio = (
        sum(m["nota"] for m in materias) / len(materias)
        if materias else 0
    )

    estadisticas.config(
        text=f"Materias: {len(materias)} | Promedio: {promedio:.2f}"
    )

def registrar():
    materia = entrada_materia.get().strip()
    profesor = entrada_profesor.get().strip()
    nota = entrada_nota.get().strip()
    fecha = entrada_fecha.get().strip()

    if not all((materia, profesor, nota, fecha)):
        messagebox.showwarning( "Campos vacíos", "Completa todos los campos."
        )
        return

    try:
        nota = float(nota)

        if not 0 <= nota <= 100:
            raise ValueError

    except ValueError:
        messagebox.showerror(  "Nota incorrecta",  "La nota debe ser un número entre 0 y 100."
        )
        return
    materias.append({  "materia": materia, "profesor": profesor, "nota": nota, 
                     "fecha": fecha, "estado": estado.get()
    })

    guardar()
    actualizar()
    limpiar()

    messagebox.showinfo(
        "Éxito",
        "Materia registrada correctamente."
    )


def eliminar():
    seleccion = tabla.selection()

    if not seleccion:
        messagebox.showwarning(
            "Seleccionar",
            "Selecciona una materia."
        )
        return

    if messagebox.askyesno(    "Confirmar","¿Deseas eliminar la materia seleccionada?"
    ):
        materias.pop(tabla.index(seleccion[0]))
        guardar()
        actualizar()


def buscar():
    texto = entrada_buscar.get().lower()

    resultados = [
        m for m in materias
        if texto in m["materia"].lower()
    ]

    actualizar(resultados)

ventana = tk.Tk()
ventana.title("Gestor de Estudios")
ventana.geometry("950x600")
ventana.resizable(False, False)

tk.Label( ventana, text="ORGANIZACION DE ESTUDIO", font=("Arial", 22, "bold")
).pack(pady=15)

frame = ttk.LabelFrame(  ventana,  text="Registrar materia"
)

frame.pack(   padx=20, fill="x"
)
campos = [ ("Materia:", 0, 0), ("Profesor:", 0, 2), ("Nota:", 1, 0), ("Fecha entrega:", 1, 2)
]
entradas = []
for texto, fila, columna in campos:
    ttk.Label(
        frame,
        text=texto
    ).grid(
        row=fila,
        column=columna,
        padx=10,
        pady=10
    )
    entrada = ttk.Entry(
        frame,
        width=25
    )
    entrada.grid(
        row=fila,
        column=columna + 1,
        padx=10
    )
    entradas.append(entrada)
entrada_materia, entrada_profesor, entrada_nota, entrada_fecha = entradas

ttk.Label(
    frame,
    text="Estado:"
).grid( row=2, column=0, padx=10, pady=10
)

estado = ttk.Combobox(
    frame,
    values=[ "Pendiente", "En progreso", "Completado"
    ],state="readonly",width=22
)

estado.grid(row=2,column=1,padx=10
)
estado.set("Pendiente")

busqueda = ttk.Frame(ventana)
busqueda.pack(padx=20,pady=10,fill="x"
)

ttk.Label( busqueda, text="Buscar materia:"
).pack(side="left")
entrada_buscar = ttk.Entry(busqueda,width=30
)
entrada_buscar.pack(   side="left", padx=10
)
ttk.Button(  busqueda,  text="Buscar",  command=buscar
).pack(side="left")

ttk.Button( busqueda, text="Mostrar todo", command=actualizar
).pack( side="left", padx=10
)
frame_tabla = ttk.Frame(ventana)
frame_tabla.pack( padx=20, fill="both", expand=True
)
columnas = ( "materia", "profesor", "nota", "fecha", "estado"
)
tabla = ttk.Treeview(frame_tabla, columns=columnas, show="headings",height=12
)
titulos = [ "Materia", "Profesor", "Nota", "Fecha entrega", "Estado"
]
anchos = [ 180, 180, 80, 150, 150
]

for columna, titulo, ancho in zip( columnas, titulos, anchos
):
    tabla.heading( columna, text=titulo
    )
    tabla.column(  columna,  width=ancho, anchor="center"
    )
tabla.pack( side="left", fill="both", expand=True
)
scroll = ttk.Scrollbar( frame_tabla, command=tabla.yview
)
scroll.pack(side="right", fill="y"
)
tabla.configure( yscrollcommand=scroll.set
)
botones = ttk.Frame(ventana)
botones.pack(pady=10)
ttk.Button( botones,text="Registrar",command=registrar
).grid( row=0, column=0, padx=5
)

ttk.Button( botones, text="Limpiar", command=limpiar
).grid( row=0, column=1, padx=5
)

ttk.Button(botones,text="Eliminar",command=eliminar
).grid(row=0, column=2, padx=5
)

estadisticas = ttk.Label(  ventana,font=("Arial", 11, "bold")
)
estadisticas.pack(pady=5)
materias = cargar()
actualizar()

ventana.mainloop()

